#### Import libraries & set seeds

In [ ]:
# Standard Libaries
import sys
from pathlib import Path

# Setting Path
notebook_path = Path().resolve()
parent_dir = notebook_path.parent
sys.path.append(str(parent_dir))
sys.path.append(str(parent_dir / "src"))

# Third Party Libraries
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import nn
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import random


# Custom Imports
from src.dataset import ImageDataset, load_image_dataframe
from src.transforms import get_transforms
from src.models.backbones import CaformerBackbone


# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pl.seed_everything(seed, workers=True, verbose=False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(15)

#### Load model

In [ ]:
class CADeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CaformerBackbone(weights_path=parent_dir / "model_weights/backbone/ca_s18_finetuned.pth")

    def forward(self, x):
        cls, features = self.backbone(x)
        return cls, features


model = CADeModel()

#### Load data

In [ ]:
dataframe_train = load_image_dataframe(data_path=notebook_path.parent / "data/iqa", split="train")

train_dataset = ImageDataset(
    dataframe=dataframe_train,
    transform=get_transforms(train=False),
    preprocess=True,
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

#### Extract features

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

all_cls = []
# Create a list of 4 lists to hold features per level
all_features = [[] for _ in range(4)]

all_labels = {"mucosal_cleaning": [], "expansion": [], "oiq": [], "retrograde": []}

with torch.no_grad():
    for batch in train_loader:
        images, mucosal_cleaning, expansion, oiq, retro = batch
        images = images.to(device)

        cls, features = model(images)  # features: list of 4 tensors

        all_cls.append(cls.cpu())

        for i, f in enumerate(features):
            all_features[i].append(f.cpu())

        all_labels["mucosal_cleaning"].append(mucosal_cleaning)
        all_labels["expansion"].append(expansion)
        all_labels["oiq"].append(oiq)
        all_labels["retrograde"].append(retro)

# Concatenate cls and each feature level
all_cls_tensor = torch.cat(all_cls, dim=0)
all_features_tensors = [torch.cat(level, dim=0) for level in all_features]

for key in all_labels:
    all_labels[key] = torch.cat(all_labels[key], dim=0)

print(f"Stored {all_cls_tensor.shape[0]} samples of cls and features across 4 levels.")

In [ ]:
feature_1 = all_features_tensors[0]
feature_2 = all_features_tensors[1]
feature_3 = all_features_tensors[2]
feature_4 = all_features_tensors[3]

#### Compute and visualize UMAP embeddings

In [ ]:
from umap import UMAP


def compute_umap_embeddings(features, normalize=True):
    if features.dim() == 4:
        features = torch.nn.functional.adaptive_avg_pool2d(features, 1).squeeze(-1).squeeze(-1)

    features_np = features.detach().cpu().numpy()
    reducer = UMAP(n_neighbors=15, n_components=2, metric="euclidean", min_dist=0.1, random_state=42)
    embeddings = reducer.fit_transform(features_np)

    if normalize:
        embeddings -= embeddings.mean(axis=0)
        embeddings /= np.max(np.abs(embeddings), axis=0)

    return embeddings


embeddings_feature4 = compute_umap_embeddings(feature_4)
embeddings_feature3 = compute_umap_embeddings(feature_3)
embeddings_feature2 = compute_umap_embeddings(feature_2)
embeddings_feature1 = compute_umap_embeddings(feature_1)

In [ ]:
def plot_embedding(
    embeddings,
    labels,
    label_name,
    figsize=(8, 8),
    dpi=300,
    save_path=None,
    show_background=False,
    show_xaxis=True,
    show_ylabel=True,
):
    assert embeddings.ndim == 2 and embeddings.shape[1] == 2, "embeddings must be (N,2)"
    labels_np = labels.detach().cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)
    assert embeddings.shape[0] == labels_np.shape[0], "len(labels) must match embeddings.shape[0]"

    valid = labels_np != -100
    labels_np = labels_np[valid]
    embeddings_valid = embeddings[valid]

    is_retrograde_mode = label_name.lower() == "retrograde" and set(np.unique(labels_np)).issubset({0, 1})

    if is_retrograde_mode:
        color_map = {
            0: ("Forward", "blue"),
            1: ("Retrograde", "darkorange"),
        }
    else:
        color_map = {
            0: ("Poor", "red"),
            1: ("Adequate", "orange"),
            2: ("Good", "green"),
        }

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    if show_background:
        ax.scatter(embeddings[:, 0], embeddings[:, 1], s=20, c="lightgray", alpha=0.45, edgecolors="none")

    for label, (name, color) in color_map.items():
        mask = labels_np == label
        if np.any(mask):
            ax.scatter(
                embeddings_valid[mask, 0],
                embeddings_valid[mask, 1],
                c=color,
                label=name,
                s=20,
                alpha=0.8,
                edgecolors="none",
            )

    tick_positions = np.linspace(-1.0, 1.0, 5)
    if show_xaxis:
        ax.set_xlabel(f"UMAP Dimension 1", fontsize=18)
        ax.set_xticks(tick_positions)
    else:
        ax.set_xlabel("")
        ax.set_xticks([])

    if show_ylabel:
        ax.set_ylabel(f"UMAP Dimension 2", fontsize=18)
        ax.set_yticks(tick_positions)
    else:
        ax.set_ylabel("")
        ax.set_yticks([])

    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")

    ax.tick_params(labelsize=16)

    # legend_title = label_name.upper() if label_name.lower() == "oiq" else label_name.capitalize()

    ax.legend(title=None, fontsize=18, title_fontsize=20, markerscale=2)

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    else:
        plt.show()


plot_embedding(
    embeddings_feature4,
    all_labels["mucosal_cleaning"],
    label_name="Mucosal Cleaning",
    save_path="umap_feature4_mucosal_cleaning.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature3,
    all_labels["mucosal_cleaning"],
    label_name="Mucosal Cleaning",
    save_path="umap_feature3_mucosal_cleaning.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature2,
    all_labels["mucosal_cleaning"],
    label_name="Mucosal Cleaning",
    save_path="umap_feature2_mucosal_cleaning.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature1,
    all_labels["mucosal_cleaning"],
    label_name="Mucosal Cleaning",
    save_path="umap_feature1_mucosal_cleaning.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=True,
)

plot_embedding(
    embeddings_feature4,
    all_labels["expansion"],
    label_name="Expansion",
    save_path="umap_feature4_expansion.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature3,
    all_labels["expansion"],
    label_name="Expansion",
    save_path="umap_feature3_expansion.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature2,
    all_labels["expansion"],
    label_name="Expansion",
    save_path="umap_feature2_expansion.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature1,
    all_labels["expansion"],
    label_name="Expansion",
    save_path="umap_feature1_expansion.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=True,
)
plot_embedding(
    embeddings_feature4,
    all_labels["oiq"],
    label_name="OIQ",
    save_path="umap_feature4_oiq.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature3,
    all_labels["oiq"],
    label_name="OIQ",
    save_path="umap_feature3_oiq.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature2,
    all_labels["oiq"],
    label_name="OIQ",
    save_path="umap_feature2_oiq.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature1,
    all_labels["oiq"],
    label_name="OIQ",
    save_path="umap_feature1_oiq.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=True,
)
plot_embedding(
    embeddings_feature4,
    all_labels["retrograde"],
    label_name="Retrograde",
    save_path="umap_feature4_retrograde.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature3,
    all_labels["retrograde"],
    label_name="Retrograde",
    save_path="umap_feature3_retrograde.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature2,
    all_labels["retrograde"],
    label_name="Retrograde",
    save_path="umap_feature2_retrograde.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=False,
)
plot_embedding(
    embeddings_feature1,
    all_labels["retrograde"],
    label_name="Retrograde",
    save_path="umap_feature1_retrograde.png",
    show_background=True,
    show_xaxis=True,
    show_ylabel=True,
)